### 사용 전, 반드시 rds_vdb_qna.py의 import 부분 변경
<주피터쓸때>
```
from rds_db_setup import get_or_create_async_db_engine
```
<모듈로 쓸때>
```
from .rds_db_setup import get_or_create_async_db_engine
```

In [2]:
import os, sys
from pathlib import Path

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_postgres.vectorstores import PGVector

from sqlalchemy import text

project_root = Path.cwd().parent

# 파이썬 모듈 검색 경로에 프로젝트 루트를 추가합니다.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
                    
from utils.rds_db_setup import get_or_create_async_db_engine
from utils.rds_vdb_qna import all_pipeline

load_dotenv()
os.environ["LANGCHAIN_PROJECT"] = "HY RAG Project"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(temperature=0, model="gpt-5-mini") 

db_engine = await get_or_create_async_db_engine("hy_rag_db",False)
DB_NAME = "hy_rag_db"

# qna 데이터 저장
await all_pipeline()

qna_table = "qna_documents"
qna_vectorstore = PGVector(
    embeddings=embeddings,
    collection_name=qna_table,
    connection=db_engine,   
)

'hy_rag_db' 데이터베이스에 'vector' 확장이 활성화되었습니다.
전처리 완료: 총 15개의 문서를 준비했습니다.
중복 검사 완료: 0개의 새로운 문서를 찾았습니다.
추가할 새로운 문서가 없습니다.

--- 모든 작업이 완료되었습니다. ---


In [14]:
'''
문서 조회하는 방법
'''
async with db_engine.connect() as connection:
    # 1. 컬렉션 ID 가져오기
    collection_query = text("SELECT uuid FROM langchain_pg_collection WHERE name = :coll_name")
    
    result = await connection.execute(collection_query, {"coll_name": qna_table})
    collection_id = result.scalar_one_or_none()
    
    # 2. 해당 컬렉션 ID를 가진 데이터 가져오기
    if collection_id:
        fetch_query = text("SELECT * FROM langchain_pg_embedding WHERE collection_id = :coll_id")
        rows_result = await connection.execute(fetch_query, {"coll_id": collection_id})
        rows = rows_result.mappings().all()
        print(f"✅ 총 {len(rows)}개의 문서(Q&A)가 Vector Store에 저장되어 있습니다.")
        print('-'*100+'\n')

        # langchain_pg_embedding의 내부 Column값 (상위 2개)
        for row in rows[:2]:
            # 임베딩 벡터는 너무 길어서 앞 50자만 출력합니다.
            embedding_sample = str(row['embedding'])[:50]
            
            print(f"🟢 ID:            {row['id']}")
            print(f"🟢 Collection_id: {row['collection_id']}")
            print(f"📄 document: \n   {row['document'][:100]}...")
            print(f"🗂️ Metadata:      {row['cmetadata']}")
            print(f"🗂️ Embedding:     {embedding_sample}...\n")
            print('-'*100)
    else:
        print(f"⚠️ 컬렉션 '{qna_table}'을 찾을 수 없습니다.")


ProgrammingError: (psycopg.errors.UndefinedTable) relation "langchain_pg_collection" does not exist
LINE 1: SELECT uuid FROM langchain_pg_collection WHERE name = $1
                         ^
[SQL: SELECT uuid FROM langchain_pg_collection WHERE name = %(coll_name)s]
[parameters: {'coll_name': 'qna_documents'}]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [5]:
def collect_categories(rows, start=1, end=3, key_prefix="category"):
    out = {}
    for i in range(start, end + 1):
        key = f"{key_prefix}{i}"
        seen, vals = set(), []
        for row in rows:
            cm = row.get("cmetadata")
            if not cm:
                continue
            v = cm.get(key)
            if v is None:
                continue
            if v not in seen:
                seen.add(v)
                vals.append(v)
        print(vals)
        out[key] = vals
    return out

result = collect_categories(rows, start=1, end=3)

['전형이해', '지원자격', '선발방법', '서류 준비', '수시전형']
['3년, 12년특례', '12년특례', '해외고선발수시']
['3년, 12년특례 비교', '지원횟수제한', '전공선택', '추가합격, 충원', '모집요강', '추합, 충원', '국내체류', '월반, 조기졸업', '평가방법', '학적서류', '제출서류']


In [9]:
'''
문서 삭제하는 방법
'''
async with db_engine.connect() as connection:
    # 'async with' 트랜잭션 블록은 성공 시 자동 커밋, 오류 시 자동 롤백됩니다.
    async with connection.begin() as transaction:
        try:
            # 삭제할 컬렉션의 고유 ID(uuid)를 찾습니다.
            collection_query = text("SELECT uuid FROM langchain_pg_collection WHERE name = :coll_name")
            
            result = await connection.execute(collection_query, {"coll_name": qna_table})
            collection_id = result.scalar_one_or_none()

            if collection_id:
                # 2. langchain_pg_embedding 테이블에서 해당 ID를 가진 모든 문서를 삭제합니다.
                delete_embeddings_query = text("DELETE FROM langchain_pg_embedding WHERE collection_id = :coll_id")
                
                del_embed_result = await connection.execute(delete_embeddings_query, {"coll_id": collection_id})
                print(f"🔹 {del_embed_result.rowcount}개의 문서(embedding) 데이터를 삭제했습니다.")

                # 3. langchain_pg_collection 테이블에서 컬렉션 자체의 기록을 삭제합니다.
                delete_collection_query = text("DELETE FROM langchain_pg_collection WHERE uuid = :coll_id")
                
                del_coll_result = await connection.execute(delete_collection_query, {"coll_id": collection_id})
                print(f"🔹 {del_coll_result.rowcount}개의 컬렉션(collection) 기록을 삭제했습니다.")
                
                print(f"\n✅ 성공! '{qna_table}' 컬렉션과 관련된 모든 데이터가 삭제되었습니다. (자동 커밋)")
            else:
                print(f"⚠️ '{qna_table}' 컬렉션을 찾을 수 없습니다. 이미 삭제되었을 수 있습니다.")

        except Exception as e:
            # 중간에 오류가 발생하면 'async with' 블록이 자동으로 롤백합니다.
            print(f"❌ 데이터 삭제 중 오류가 발생하여 모든 변경사항을 되돌립니다 (자동 롤백): {e}")

⚠️ 'qna_documents' 컬렉션을 찾을 수 없습니다. 이미 삭제되었을 수 있습니다.


In [13]:
# 테이블 조회
async with db_engine.connect() as conn:
    # public 스키마 모든 테이블
    tables = (await conn.execute(text("""
        SELECT tablename 
        FROM pg_tables 
        WHERE schemaname='public'
        ORDER BY tablename
    """))).scalars().all()
    print("public 테이블:", tables)

    # vector 컬럼 보유 테이블(pgvector)
    vector_tables = (await conn.execute(text("""
        SELECT DISTINCT table_schema, table_name
        FROM information_schema.columns
        WHERE udt_name = 'vector'
        ORDER BY table_schema, table_name
    """))).all()
    print("vector 컬럼 테이블:", vector_tables)


public 테이블: ['car_documents', 'qna_chat_history', 'qna_summary']
vector 컬럼 테이블: []


In [ ]:
# 테이블 삭제
tables_to_drop = ["langchain_pg_embedding"]  # 삭제할 테이블 넣기

async with db_engine.begin() as conn:
    for t in tables_to_drop:
        await conn.execute(text(f"""
            DO $$
            BEGIN
              IF to_regclass('public.{t}') IS NOT NULL THEN
                EXECUTE 'DROP TABLE {t} CASCADE';
              END IF;
            END
            $$;
        """))
